# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [14]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

# Target proxy: down vs not-down
y = (df["trend_direction"] == "down").astype(int)

# Conservative initial feature set:
# Keep content metadata only; exclude IDs, target/trend fields,
# performance metrics and fields with unverified timing.
numeric_features = [
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "content_type",
    "main_intent",
]

# Keep only columns that actually exist in the dataset
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X = df[numeric_features + categorical_features].copy()

# Treat zero word/character counts as missing only if source documentation
# confirms zero means unavailable; otherwise preserve the original values.
# Impute numeric missing values using medians.
for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

# Fill missing categories explicitly
for col in categorical_features:
    X[col] = X[col].fillna("Unknown").astype(str)

# One-hot encode categorical fields
X = pd.get_dummies(
    X,
    columns=categorical_features,
    dummy_na=False,
    dtype=int
)

print("Feature vector shape:", X.shape)
print("Target distribution:")
print(y.value_counts())
print("\nFeature columns:")
print(X.columns.tolist())

assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("\nFeature vector built successfully.")


Feature vector shape: (30000, 12)
Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Feature columns:
['word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_Unknown', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional']

Feature vector built successfully.


In [15]:

!git clone https://github.com/vishalseries7030/flyrank-ml-internship.git /content/flyrank-ml-internship

!ls -la /content/flyrank-ml-internship
!find /content/flyrank-ml-internship -iname "*content_refresh*"


fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.
total 104
drwxr-xr-x 12 root root  4096 Oct 21 09:31 .
drwxr-xr-x  1 root root  4096 Oct 21 09:31 ..
-rw-r--r--  1 root root   654 Oct 21 09:31 AGENTS.md
-rw-r--r--  1 root root   654 Oct 21 09:31 CLAUDE.md
drwxr-xr-x  3 root root  4096 Oct 21 09:31 data
-rw-r--r--  1 root root  2763 Oct 21 09:31 DATA_USE.md
drwxr-xr-x  3 root root  4096 Oct 21 09:31 docs
drwxr-xr-x  8 root root  4096 Oct 21 09:31 .git
drwxr-xr-x  3 root root  4096 Oct 21 09:31 .github
-rw-r--r--  1 root root   993 Oct 21 09:31 .gitignore
-rw-r--r--  1 root root 10408 Oct 21 09:31 GUIDE.md
-rw-r--r--  1 root root  1289 Oct 21 09:31 LICENSE
drwxr-xr-x  2 root root  4096 Oct 21 09:31 notebooks
drwxr-xr-x  3 root root  4096 Oct 21 09:31 outputs
-rw-r--r--  1 root root  9847 Oct 21 09:31 README.md
-rw-r--r--  1 root root   107 Oct 21 09:31 requirements.txt
drwxr-xr-x  2 root root  4096 Oct 21 09:31 scripts
-rw-r--r--  1 r

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


### Feature notes

- **Search demand and competition:** `search_volume`, `competition`, `competition_level`, and `cpc` describe keyword demand, competition, and estimated cost-per-click. Numeric missing values will be imputed using the training-set median; categorical missing values will be handled as an explicit unknown category. These fields are usable only if their source snapshot is available at prediction time.

- **Content attributes:** `content_type` and `main_intent` describe content format and intent. `word_count`, `char_count`, `word_count_tier`, and `char_count_tier` describe content size. Missing text-length values will be imputed using training data, with missingness preserved where useful. Availability depends on whether the content metadata was captured before scoring.

- **Content age and freshness:** `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, and `freshness_tier` describe page age and recency. These are potentially available at scoring time if computed from timestamps known at that moment.

- **Historical performance:** `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, and `days_with_sessions` summarize activity over a 90-day window. They are safe only when that window ends before the scoring cutoff.

- **Recent and previous-period metrics:** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, and `sessions_prev_30d` describe two 30-day periods. They are excluded from the initial feature set until their exact window boundaries and availability are verified.

- **Derived engagement metrics:** `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct` summarize performance. Their formulas and source-window timing must be verified before use. Treat `avg_position == 0` as potentially unavailable rather than a genuine ranking.

- **Categorical operational metadata:** `provider_used` and `model_used` describe the provider/model used to generate content. They may be missing and may not be available or appropriate for the review decision; exclude them from the initial model pending justification.

- **Identifiers:** `content_id` and `client_id` identify content and clients. Exclude them from model features to avoid memorization and identity leakage. Use `client_id` only for grouped evaluation if needed.

- **Target and trend fields:** `trend_direction` is the proxy target for this assignment, and `trend_pct` is directly related to the trend outcome. Exclude both from the feature vector. Do not claim the proxy proves future decline.

- **Missingness summary:** `search_volume`, `competition`, and `cpc` each have 8.23% missing values; `word_count` and `char_count` each have 25.66%; `provider_used` has 71.46%; `trend_pct` has 11.29%. Missingness alone does not establish whether a field is safe or unsafe.

**Prediction-time rule:** A feature is eligible only when its value would genuinely be known at the scoring cutoff. The dataset's column names alone do not establish that timing, so window boundaries and source timestamps must be checked before training.


In [16]:

from pathlib import Path
import pandas as pd
import numpy as np

# Load the dataset
DATA_PATH = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nFeature availability and missingness audit")
print("=" * 55)

for col in df.columns:
    missing = int(df[col].isna().sum())
    missing_pct = round(df[col].isna().mean() * 100, 2)

    print(
        f"{col}: dtype={df[col].dtype}, "
        f"missing={missing} ({missing_pct}%)"
    )


Dataset shape: (30000, 44)

Feature availability and missingness audit
content_id: dtype=object, missing=0 (0.0%)
client_id: dtype=object, missing=0 (0.0%)
search_volume: dtype=float64, missing=2468 (8.23%)
competition: dtype=float64, missing=2468 (8.23%)
competition_level: dtype=object, missing=2610 (8.7%)
cpc: dtype=float64, missing=2468 (8.23%)
content_type: dtype=object, missing=0 (0.0%)
main_intent: dtype=object, missing=2374 (7.91%)
word_count: dtype=float64, missing=7699 (25.66%)
char_count: dtype=float64, missing=7699 (25.66%)
provider_used: dtype=object, missing=21438 (71.46%)
model_used: dtype=object, missing=5733 (19.11%)
impressions_90d: dtype=int64, missing=0 (0.0%)
clicks_90d: dtype=int64, missing=0 (0.0%)
pageviews_90d: dtype=int64, missing=0 (0.0%)
sessions_90d: dtype=int64, missing=0 (0.0%)
users_90d: dtype=int64, missing=0 (0.0%)
engaged_sessions_90d: dtype=int64, missing=0 (0.0%)
ai_sessions_90d: dtype=int64, missing=0 (0.0%)
scroll_events_90d: dtype=int64, missing=0

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


### Leakage hunt

The target proxy is `trend_direction`. I will exclude both `trend_direction` and `trend_pct` from model inputs because they directly describe the trend outcome.

I will exclude `content_id` and `client_id` from model features to prevent identifier memorization. `client_id` may be retained only to create client-grouped train/test splits.

The 90-day, last-30-day, and previous-30-day performance fields need a verified prediction cutoff and window boundaries before they can be used safely. Their names alone do not prove that the data was available before scoring. I will flag them for temporal verification rather than assume they are safe.

Derived metrics such as `ctr`, `engagement_rate`, `scroll_rate`, `avg_position`, and `ai_traffic_pct` also need formula and source-window checks. `avg_position == 0` may represent unavailable data, not a real position.

`provider_used` and `model_used` are excluded from the initial feature set because their availability and relevance to the review decision have not been established. Missing values in other candidate features must be imputed using training data only.

This audit identifies leakage risks; it does not prove that all remaining fields are leakage-free. No future-performance claim will be made from agreement with the historical trend proxy.


In [17]:

# Section 2: Feature notes — data-quality audit

assert "df" in globals(), "Run the dataset-loading cell first."

feature_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})

display(
    feature_audit.sort_values(
        "missing_pct", ascending=False
    )
)

print("\nIdentifiers — exclude from model features:")
print([c for c in ["content_id", "client_id"] if c in df.columns])

print("\nTarget/trend-related fields — exclude from initial features:")
print([
    c for c in ["trend_direction", "trend_pct"]
    if c in df.columns
])

print("\nWindow-sensitive fields — verify cutoff before use:")
print([
    c for c in df.columns
    if any(token in c for token in ["90d", "last_30d", "prev_30d"])
])


,dtype,missing_count,missing_pct,unique_values
provider_used,object,21438,71.46,2
word_count,float64,7699,25.66,5476
char_count,float64,7699,25.66,14839
word_count_tier,object,7699,25.66,4
char_count_tier,object,7699,25.66,4
model_used,object,5733,19.11,5
trend_pct,float64,3388,11.29,2712
competition_level,object,2610,8.70,3
search_volume,float64,2468,8.23,41
cpc,float64,2468,8.23,915



Identifiers — exclude from model features:
['content_id', 'client_id']

Target/trend-related fields — exclude from initial features:
['trend_direction', 'trend_pct']

Window-sensitive fields — verify cutoff before use:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


### What I excluded and why

| Excluded field(s) | Reason |
|---|---|
| `content_id` | Unique content identifier; can cause memorization rather than generalizable learning. |
| `client_id` | Client identifier; exclude from model features to reduce client-specific memorization. Retain only for grouped evaluation where required. |
| `trend_direction` | This is the target proxy and must never be included as an input feature. |
| `trend_pct` | Directly describes the trend outcome and risks target leakage. |
| `provider_used`, `model_used` | Excluded from the initial feature set because prediction-time availability and relevance to review prioritization have not been established. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Excluded until their reporting windows are verified to end before the prediction cutoff. |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | Excluded from the initial feature set until temporal availability and their relationship to the target are verified. |
| All `*_90d` performance fields | Held out until the 90-day reporting window is confirmed to precede the scoring cutoff. |
| `ctr`, `engagement_rate`, `scroll_rate`, `avg_position`, `ai_traffic_pct` | Derived performance metrics; exclude until formulas, source windows, and availability are verified. `avg_position == 0` may indicate unavailable data. |
| `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier` | Exclude redundant derived tiers initially; use underlying values only when their availability is verified. |
| `search_volume`, `competition`, `competition_level`, `cpc` | Hold out until the keyword data snapshot and its availability at scoring time are established. |
| `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update` | Candidate features, not automatically approved. Include only after confirming these values are available before scoring and are not derived using future information. |

This is a conservative exclusion list for the initial feature vector. A field is not safe merely because its name sounds historical. The final feature set must be consistent with the documented prediction cutoff and the target definition.


In [18]:

# Section 3: Leakage hunt — explicit checks

assert "df" in globals(), "Run the dataset-loading cell first."

target_col = "trend_direction"

identifier_cols = ["content_id", "client_id"]

target_related_cols = ["trend_direction", "trend_pct"]

window_sensitive_cols = [
    c for c in df.columns
    if any(token in c for token in ["90d", "last_30d", "prev_30d"])
]

derived_metric_cols = [
    c for c in [
        "ctr", "engagement_rate", "scroll_rate",
        "avg_position", "ai_traffic_pct"
    ]
    if c in df.columns
]

operational_cols = [
    c for c in ["provider_used", "model_used"]
    if c in df.columns
]

print("1. Target exists:", target_col in df.columns)
print("2. Target missing values:", df[target_col].isna().sum())
print("3. Target distribution:")
display(df[target_col].value_counts(dropna=False).to_frame("count"))

print("\n4. Identifier columns excluded from features:")
print([c for c in identifier_cols if c in df.columns])

print("\n5. Target/trend-related columns excluded:")
print([c for c in target_related_cols if c in df.columns])

print("\n6. Window-sensitive columns requiring cutoff verification:")
print(window_sensitive_cols)

print("\n7. Derived metrics requiring formula/window verification:")
print(derived_metric_cols)

print("\n8. Operational metadata excluded pending justification:")
print(operational_cols)

print("\n9. avg_position == 0 rows:")
if "avg_position" in df.columns:
    print(int(df["avg_position"].eq(0).sum()))

# Assertions to prevent accidental inclusion of obvious leakage columns
initial_exclusions = set(
    identifier_cols + target_related_cols + operational_cols
)
assert target_col in df.columns
assert "trend_pct" not in set(df.columns) - initial_exclusions

print("\nLeakage-risk checks completed.")
print("Note: window timing and metric formulas still require source documentation.")


1. Target exists: True
2. Target missing values: 0
3. Target distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



4. Identifier columns excluded from features:
['content_id', 'client_id']

5. Target/trend-related columns excluded:
['trend_direction', 'trend_pct']

6. Window-sensitive columns requiring cutoff verification:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

7. Derived metrics requiring formula/window verification:
['ctr', 'engagement_rate', 'scroll_rate', 'avg_position', 'ai_traffic_pct']

8. Operational metadata excluded pending justification:
['provider_used', 'model_used']

9. avg_position == 0 rows:
1205

Leakage-risk checks completed.
Note: window timing and metric formulas still require source documentation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.